In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'url_corpus' and cwd.parent.name == 'notebooks':
    os.chdir(cwd.parent)

nb_dir = Path.cwd().resolve()
backend_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

print(f'Notebook CWD: {Path.cwd()}')
print(f'Backend root in sys.path: {backend_root}')


Notebook CWD: /Users/maksymiliansowula/Documents/Studia/3semestrmgr/Magisterka/PhisingDetector/backend/notebooks
Backend root in sys.path: /Users/maksymiliansowula/Documents/Studia/3semestrmgr/Magisterka/PhisingDetector/backend


# Kompleksowa Analiza Statystyczna - Zbiór Danych Detekcji Phishingu


## 1. Import Bibliotek i Wczytanie Danych


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
import os
import sys

cwd = Path.cwd().resolve(); os.chdir(cwd.parent) if cwd.name == 'url_corpus' and cwd.parent.name == 'notebooks' else None; nb_dir = Path.cwd().resolve(); backend_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir; sys.path.insert(0, str(backend_root)) if str(backend_root) not in sys.path else None
from utils.feature_names import FEATURE_CATEGORIES, get_readable_name

warnings.filterwarnings('ignore')

output_dir = '../static/statistical_analysis_url_corpus'
csv_dir = f'{output_dir}/csv'
png_dir = f'{output_dir}/png'
html_dir = f'{output_dir}/html'
docs_dir = f'{output_dir}/docs'

for directory in [output_dir, csv_dir, png_dir, html_dir, docs_dir]:
    os.makedirs(directory, exist_ok=True)

px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = px.colors.sequential.Viridis
px.defaults.color_discrete_sequence = px.colors.qualitative.Vivid

dataset_path = '../data/url_corpus/dataset_full.csv'; dataset_path = dataset_path if os.path.exists(dataset_path) else '../data/url_corpus/dataset_optimized.csv'; print(f'Using dataset: {dataset_path}'); df = pd.read_csv(dataset_path)
print("✅ Dane załadowane pomyślnie!")
print(f"Rozmiar zbioru danych: {df.shape}")
print(f"Liczba cech: {df.shape[1] - 1}")
print(f"\nKategorie cech:")
for category, features in FEATURE_CATEGORIES.items():
    print(f"  • {category}: {len(features)} cech")
print(f"\nPierwszych 5 wierszy:")
df.head()


Using dataset: ../data/url_corpus/dataset_full.csv
✅ Dane załadowane pomyślnie!
Rozmiar zbioru danych: (100000, 105)
Liczba cech: 104

Kategorie cech:
  • URL Properties: 21 cech
  • Domain Properties: 20 cech
  • Directory Properties: 18 cech
  • File Properties: 18 cech
  • Parameter Properties: 22 cech
  • External/DNS Properties: 14 cech

Pierwszych 5 wierszy:


,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,qty_tld_url,length_url,...,qty_hashtag_domain,qty_dollar_domain,qty_percent_domain,domain_in_ip,server_client_domain,qty_dollar_file,tld_present_params,url_google_index,domain_google_index,phishing
0,2,0,0,5,0,0,0,0,2,52,...,0,0,0,0,0,0,0,0,0,1
1,1,0,0,4,0,0,0,0,2,34,...,0,0,0,0,0,0,0,0,0,1
2,4,2,0,6,0,0,0,0,2,73,...,0,0,0,0,0,0,0,0,0,1
3,1,2,0,4,0,0,0,0,2,36,...,0,0,0,0,0,0,0,0,0,1
4,2,0,0,5,0,0,0,0,2,49,...,0,0,0,0,0,0,0,0,0,1


## 2. Przetwarzanie Wstępne - Czyszczenie i Obsługa Brakujących Danych


In [3]:
def clean_and_analyze_data(df):
    print("=" * 80)
    print("ANALIZA JAKOŚCI DANYCH")
    print("=" * 80)

    print("\n1. PRZEGLĄD ZBIORU DANYCH:")
    print(f"   • Wiersze: {df.shape[0]:,}")
    print(f"   • Kolumny: {df.shape[1]}")
    print(f"   • Cechy: {df.shape[1] - 1}")
    print(f"   • Zmienna docelowa: {df.columns[-1]}")

    print("\n2. TYPY DANYCH:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"   • {dtype}: {count} kolumn")

    print("\n3. ANALIZA BRAKUJĄCYCH WARTOŚCI:")
    missing_values = df.isnull().sum()
    if missing_values.sum() == 0:
        print("   ✅ Nie znaleziono brakujących wartości!")
    else:
        missing_percentage = (missing_values / len(df)) * 100
        missing_df = pd.DataFrame({
            'Kolumna': missing_values.index, 'Brakujące': missing_values.values, 'Procent': missing_percentage.values
        })
        print("   ⚠️  Wykryto brakujące wartości:")
        print(missing_df[missing_df['Brakujące'] > 0])

    print("\n4. ANALIZA DUPLIKATÓW:")
    duplicates = df.duplicated().sum()
    print(f"   • Zduplikowane wiersze: {duplicates} ({duplicates/len(df)*100:.2f}%)")

    print("\n5. ROZKŁAD ZMIENNEJ DOCELOWEJ:")
    target_col = df.columns[-1]
    target_counts = df[target_col].value_counts().sort_index()

    for value, count in target_counts.items():
        percentage = (count / len(df)) * 100
        label = "Brak phishingu" if value == 0 else "Phishing"
        print(f"   • {label} (klasa {value}): {count:,} próbek ({percentage:.1f}%)")

    class_ratio = target_counts.min() / target_counts.max()
    print(f"\n   Współczynnik balansu klas: {class_ratio:.2f}")
    if class_ratio < 0.5:
        print("   ⚠️  Zbiór jest niezbalansowany")
    else:
        print("   ✅ Zbiór jest rozsądnie zbalansowany")

    df_clean = df.copy()
    if duplicates > 0:
        df_clean = df_clean.drop_duplicates()
        print(f"\n6. AKCJE CZYSZCZENIA:")
        print(f"   ✓ Usunięto {duplicates} zduplikowanych wierszy")

    print("\n" + "=" * 80)
    print("CZYSZCZENIE DANYCH ZAKOŃCZONE")
    print("=" * 80)
    print(f"Końcowy rozmiar zbioru danych: {df_clean.shape}")

    return df_clean

df_clean = clean_and_analyze_data(df)

ANALIZA JAKOŚCI DANYCH

1. PRZEGLĄD ZBIORU DANYCH:
   • Wiersze: 100,000
   • Kolumny: 105
   • Cechy: 104
   • Zmienna docelowa: phishing

2. TYPY DANYCH:
   • int64: 104 kolumn
   • float64: 1 kolumn

3. ANALIZA BRAKUJĄCYCH WARTOŚCI:
   ✅ Nie znaleziono brakujących wartości!

4. ANALIZA DUPLIKATÓW:
   • Zduplikowane wiersze: 18987 (18.99%)

5. ROZKŁAD ZMIENNEJ DOCELOWEJ:
   • Brak phishingu (klasa 0): 50,000 próbek (50.0%)
   • Phishing (klasa 1): 50,000 próbek (50.0%)

   Współczynnik balansu klas: 1.00
   ✅ Zbiór jest rozsądnie zbalansowany

6. AKCJE CZYSZCZENIA:
   ✓ Usunięto 18987 zduplikowanych wierszy

CZYSZCZENIE DANYCH ZAKOŃCZONE
Końcowy rozmiar zbioru danych: (81013, 105)


## 3. Statystyki Opisowe


In [4]:
def calculate_descriptive_statistics(df):
    print("=" * 80)
    print("ANALIZA STATYSTYK OPISOWYCH")
    print("=" * 80)

    numeric_columns = df.select_dtypes(include=[np.number]).columns[:-1]
    stats_df = df[numeric_columns].describe().transpose()

    stats_df['zakres'] = stats_df['max'] - stats_df['min']
    stats_df['IQR'] = stats_df['75%'] - stats_df['25%']
    stats_df['skew'] = df[numeric_columns].skew()
    stats_df['kurtosis'] = df[numeric_columns].kurt()

    stats_df = stats_df.rename(columns={
        'mean': 'srednia',
        'std': 'odch_std',
        '25%': 'Q1',
        '50%': 'mediana',
        '75%': 'Q3'
    })

    print("\nPODSTAWOWE STATYSTYKI (Top 20 cech):")
    print("=" * 80)
    display_stats = stats_df[['min', 'max', 'srednia', 'odch_std', 'mediana', 'Q1', 'Q3', 'zakres', 'IQR']].head(20).round(3).copy()
    display_stats.index = [get_readable_name(feat) for feat in display_stats.index]
    print(display_stats)

    print("\nKSZTAŁT ROZKŁADU (Top 20 cech):")
    print("=" * 80)
    shape_stats = stats_df[['skew', 'kurtosis']].head(20).round(3).copy()
    shape_stats.index = [get_readable_name(feat) for feat in shape_stats.index]
    print(shape_stats)

    stats_df.round(4).to_csv(f'{csv_dir}/descriptive_statistics.csv')
    print(f"\n✓ Statystyki zapisane do: {csv_dir}/descriptive_statistics.csv")

    print("\nINTERPRETACJA ROZKŁADU:")
    print("=" * 80)
    highly_skewed = stats_df[abs(stats_df['skew']) > 1].index.tolist()
    highly_kurtotic = stats_df[abs(stats_df['kurtosis']) > 3].index.tolist()
    print(f"   • Silnie skośne cechy (|skośność| > 1): {len(highly_skewed)}")
    if highly_skewed:
        readable_skewed = [get_readable_name(feat) for feat in highly_skewed[:5]]
        print(f"     Przykłady: {', '.join(readable_skewed)}")
    print(f"   • Silnie kurtyczne cechy (|kurtoza| > 3): {len(highly_kurtotic)}")
    if highly_kurtotic:
        readable_kurtotic = [get_readable_name(feat) for feat in highly_kurtotic[:5]]
        print(f"     Przykłady: {', '.join(readable_kurtotic)}")

    return stats_df

descriptive_stats = calculate_descriptive_statistics(df_clean)

ANALIZA STATYSTYK OPISOWYCH

PODSTAWOWE STATYSTYKI (Top 20 cech):
                                min     max  srednia  odch_std  mediana    Q1  \
URL: Liczba "." w całym URL     1.0   255.0    1.973     2.056      1.0   1.0   
URL: Liczba "-" w całym URL     0.0    64.0    0.486     1.257      0.0   0.0   
URL: Liczba "_" w całym URL     0.0    38.0    0.227     1.136      0.0   0.0   
URL: Liczba "/" w całym URL     2.0   161.0    4.152     2.171      3.0   3.0   
URL: Liczba "?" w całym URL     0.0    64.0    0.165     0.537      0.0   0.0   
URL: Liczba "=" w całym URL     0.0    64.0    0.409     1.334      0.0   0.0   
URL: Liczba "@" w całym URL     0.0    32.0    0.019     0.240      0.0   0.0   
URL: Liczba "&" w całym URL     0.0    24.0    0.216     1.046      0.0   0.0   
URL: Długość Top Level Domain   0.0   129.0    1.491     1.091      2.0   1.0   
URL: Całkowita długość         13.0  4388.0   60.176    80.889     27.0  20.0   
Domena: Liczba "."              0.0     3.0

## 4. Analiza Korelacji


In [5]:
def analyze_correlations(df):
    print("=" * 80)
    print("ANALIZA KORELACJI")
    print("=" * 80)

    target_col = df.columns[-1]
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col)

    print(f"\n1. KORELACJE ZE ZMIENNĄ DOCELOWĄ ({target_col}):")
    target_correlations = df[feature_cols].corrwith(df[target_col]).to_frame('Korelacja')
    target_correlations['Abs_Korelacja'] = target_correlations['Korelacja'].abs()
    target_correlations = target_correlations.sort_values('Abs_Korelacja', ascending=False)

    print(f"\nTop 20 Cech Najbardziej Skorelowanych z '{target_col}':")
    top_20_corr = target_correlations.head(20).copy()
    top_20_corr.index = [get_readable_name(feat) for feat in top_20_corr.index]
    print(top_20_corr.to_string())

    top_30_data = target_correlations.head(30).sort_values('Korelacja').copy()
    top_30_data.index = [get_readable_name(feat) for feat in top_30_data.index]

    fig = px.bar(top_30_data,
                 y='Korelacja',
                 title=f'Top 30 Cech Skorelowanych z "{target_col}" (Phishing)',
                 labels={'index': 'Cecha', 'Korelacja': 'Współczynnik korelacji Pearsona'},
                 color='Korelacja',
                 color_continuous_scale='RdBu_r',
                 range_color=[-1, 1])
    fig.update_layout(height=700)
    fig.write_html(f'{html_dir}/target_correlations.html')
    fig.write_image(f'{png_dir}/target_correlations.png', scale=2)
    fig.show()

    print("\n2. ANALIZA WIELOKOLINEARNOŚCI:")
    correlation_matrix = df[feature_cols].corr()
    high_corr_pairs = correlation_matrix.unstack().sort_values(ascending=False)
    high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.7]
    high_corr_pairs = high_corr_pairs[high_corr_pairs < 1.0]

    if not high_corr_pairs.empty:
        print(f"   Znaleziono {len(high_corr_pairs)} wysoko skorelowanych par cech (|r| > 0.7)")
        print("\n   Top 10 skorelowanych par:")
        top_10_pairs = high_corr_pairs.head(10).copy()
        top_10_pairs.index = [(get_readable_name(idx[0]), get_readable_name(idx[1])) for idx in top_10_pairs.index]
        print(top_10_pairs)
    else:
        print("   ✅ Nie wykryto wysokiej wielokolinearności (|r| > 0.7)")

    target_correlations.to_csv(f'{csv_dir}/target_correlations.csv')
    print(f"\n✓ Analiza korelacji zapisana do: {csv_dir}/target_correlations.csv")
    print(f"✓ Wizualizacja zapisana do: {png_dir}/target_correlations.png")

    return target_correlations, correlation_matrix

target_correlations, correlation_matrix = analyze_correlations(df_clean)


ANALIZA KORELACJI

1. KORELACJE ZE ZMIENNĄ DOCELOWĄ (phishing):

Top 20 Cech Najbardziej Skorelowanych z 'phishing':
                              Korelacja  Abs_Korelacja
URL: Liczba "/" w całym URL    0.581580       0.581580
Ścieżka: Liczba "/"            0.580345       0.580345
Plik: Liczba "."               0.575783       0.575783
URL: Całkowita długość         0.527017       0.527017
SSL: Czy ma ważny certyfikat  -0.513359       0.513359
URL: Liczba "." w całym URL    0.472872       0.472872
Ścieżka: Długość               0.418347       0.418347
Plik: Długość nazwy            0.375394       0.375394
URL: Liczba "-" w całym URL    0.356460       0.356460
URL: Liczba "?" w całym URL    0.336809       0.336809
URL: Liczba "=" w całym URL    0.335510       0.335510
HTTP: Liczba przekierowań     -0.324195       0.324195
Parametry: Liczba parametrów   0.306063       0.306063
DNS: Liczba serwerów MX       -0.289570       0.289570
Parametry: Długość             0.275893       0.275893
Dom


2. ANALIZA WIELOKOLINEARNOŚCI:
   Znaleziono 40 wysoko skorelowanych par cech (|r| > 0.7)

   Top 10 skorelowanych par:
(Parametry: Liczba "*", URL: Liczba "*" w całym URL)           0.989076
(URL: Liczba "*" w całym URL, Parametry: Liczba "*")           0.989076
(Parametry: Liczba "&", URL: Liczba "&" w całym URL)           0.977571
(URL: Liczba "&" w całym URL, Parametry: Liczba "&")           0.977571
(URL: Liczba "/" w całym URL, Ścieżka: Liczba "/")             0.935890
(Ścieżka: Liczba "/", URL: Liczba "/" w całym URL)             0.935890
(Parametry: Liczba parametrów, URL: Liczba "=" w całym URL)    0.876910
(URL: Liczba "=" w całym URL, Parametry: Liczba parametrów)    0.876910
(Parametry: Liczba "&", Parametry: Liczba parametrów)          0.866214
(Parametry: Liczba parametrów, Parametry: Liczba "&")          0.866214
dtype: float64

✓ Analiza korelacji zapisana do: ../static/statistical_analysis_url_corpus/csv/target_correlations.csv
✓ Wizualizacja zapisana do: ../static/st

## 5. Analiza Ważności Cech (Random Forest)


In [6]:
def calculate_feature_importance(df, n_estimators=100):
    print("=" * 80)
    print("ANALIZA WAŻNOŚCI CECH (RANDOM FOREST)")
    print("=" * 80)

    target_col = df.columns[-1]
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col)

    X = df[feature_cols]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    print(f"\n1. TRENOWANIE RANDOM FOREST:")
    print(f"   • Liczba drzew: {n_estimators}")
    print(f"   • Zbiór treningowy: {X_train.shape[0]:,} próbek")
    print(f"   • Zbiór testowy: {X_test.shape[0]:,} próbek")

    rf_model = RandomForestClassifier(n_estimators=n_estimators, random_state=42, n_jobs=-1, max_depth=10)
    rf_model.fit(X_train, y_train)

    train_score = rf_model.score(X_train, y_train)
    test_score = rf_model.score(X_test, y_test)
    print(f"\n   • Dokładność na zbiorze treningowym: {train_score:.4f}")
    print(f"\n   • Dokładność na zbiorze testowym: {test_score:.4f}")

    print("\n2. WAŻNOŚĆ CECH:")
    importances = rf_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Cecha': feature_cols,
        'Ważność': importances
    }).sort_values('Ważność', ascending=False)

    print("\nTop 20 Najważniejszych Cech:")
    top_20_importance = feature_importance_df.head(20).copy()
    top_20_importance['Cecha'] = top_20_importance['Cecha'].apply(get_readable_name)
    print(top_20_importance.to_string(index=False))

    top_30_importance = feature_importance_df.head(30).sort_values('Ważność', ascending=True).copy()
    top_30_importance['Cecha'] = top_30_importance['Cecha'].apply(get_readable_name)

    fig = px.bar(
        top_30_importance,
        x='Ważność',
        y='Cecha',
        orientation='h',
        title='Top 30 Najważniejszych Cech (Random Forest)',
        labels={'Ważność': 'Ważność cechy', 'Cecha': 'Nazwa cechy'},
        color='Ważność',
        color_continuous_scale='RdYlGn_r'
    )
    fig.update_layout(height=800, yaxis={'categoryorder': 'total ascending'})
    fig.write_html(f'{html_dir}/feature_importance.html')
    fig.write_image(f'{png_dir}/feature_importance.png', scale=2)
    fig.show()

    feature_importance_df.to_csv(f'{csv_dir}/feature_importance.csv', index=False)
    print(f"\n✓ Ważność cech zapisana do: {csv_dir}/feature_importance.csv")
    print(f"✓ Wizualizacja zapisana do: {png_dir}/feature_importance.png")

    return feature_importance_df

feature_importance = calculate_feature_importance(df_clean)


ANALIZA WAŻNOŚCI CECH (RANDOM FOREST)

1. TRENOWANIE RANDOM FOREST:
   • Liczba drzew: 100
   • Zbiór treningowy: 56,709 próbek
   • Zbiór testowy: 24,304 próbek

   • Dokładność na zbiorze treningowym: 0.9999

   • Dokładność na zbiorze testowym: 0.9999

2. WAŻNOŚĆ CECH:

Top 20 Najważniejszych Cech:
                       Cecha  Ważność
         Plik: Długość nazwy 0.217666
      URL: Całkowita długość 0.182068
 URL: Liczba "." w całym URL 0.101928
 URL: Liczba "/" w całym URL 0.099551
         Ścieżka: Liczba "/" 0.092571
            Ścieżka: Długość 0.063993
            Plik: Liczba "." 0.047792
 URL: Liczba "=" w całym URL 0.027149
SSL: Czy ma ważny certyfikat 0.026367
        DNS: Czas odpowiedzi 0.018647
 URL: Liczba "-" w całym URL 0.016248
          Parametry: Długość 0.014681
 URL: Liczba "?" w całym URL 0.013394
Parametry: Liczba parametrów 0.011701
           DNS: TTL hostname 0.008886
 URL: Liczba "_" w całym URL 0.007155
          Domena: Liczba "." 0.006803
     DNS: Lic


✓ Ważność cech zapisana do: ../static/statistical_analysis_url_corpus/csv/feature_importance.csv
✓ Wizualizacja zapisana do: ../static/statistical_analysis_url_corpus/png/feature_importance.png


## 6. Analiza Rozkładów i Testy Normalności


### Wyjaśnienie wartości -1 w zbiorze danych

Dataset używa **-1 jako wartości oznaczającej "nie występuje w URL"**:

- **Cechy directory/file/params**: -1 = URL nie zawiera ścieżki/pliku/parametrów
- **Cechy DNS/WHOIS**: -1 = informacja niedostępna (timeout, błąd)
- **Cechy binarne**: -1 = nie można ustalić (np. domain_spf, url_google_index)

To **NIE są dane przeskalowane** - to konwencja oznaczająca nieobecność cechy w URL.


In [7]:
def analyze_distributions(df, sample_size=1000):
    print("=" * 80)
    print("ANALIZA ROZKŁADÓW")
    print("=" * 80)

    numeric_columns = df.select_dtypes(include=[np.number]).columns[:-1]
    normality_results = []

    print("\nSprawdzanie wartości -1 (nie występuje w URL):")
    features_with_minus_one = {}
    for column in numeric_columns:
        count_minus_one = (df[column] == -1).sum()
        if count_minus_one > 0:
            features_with_minus_one[column] = count_minus_one

    print(f"   • Znaleziono {len(features_with_minus_one)} cech z wartościami -1")
    print(f"   • Wartość -1 oznacza: cecha nie występuje w URL")

    for column in numeric_columns:
        sample_data = df[column].sample(min(len(df), sample_size), random_state=42)
        statistic, p_value = stats.shapiro(sample_data)
        normality_results.append({
            'Cecha': column,
            'Statystyka Shapiro-Wilk': statistic,
            'wartość-p': p_value,
            'Rozkład Normalny': p_value > 0.05
        })

    normality_df = pd.DataFrame(normality_results)
    normal_count = normality_df['Rozkład Normalny'].sum()
    print(f"\nWyniki Testu Normalności (Shapiro-Wilk, α=0.05):")
    print(f"   • Cechy z rozkładem normalnym: {normal_count} ({normal_count/len(normality_df)*100:.1f}%)")
    print(f"   • Cechy z rozkładem nienormalnym: {len(normality_df) - normal_count}")

    top_features = feature_importance['Cecha'].head(8).tolist()
    readable_titles = [get_readable_name(f) for f in top_features]

    fig = make_subplots(
        rows=2, cols=4,
        subplot_titles=readable_titles,
        vertical_spacing=0.15,
        horizontal_spacing=0.08
    )

    colors = ['#636EFA', '#EF553B']

    for i, feature in enumerate(top_features):
        row, col = i // 4 + 1, i % 4 + 1

        data0_all = df[df['phishing'] == 0][feature]
        data1_all = df[df['phishing'] == 1][feature]

        data0 = data0_all[data0_all != -1].dropna().values
        data1 = data1_all[data1_all != -1].dropna().values

        excluded0 = (data0_all == -1).sum()
        excluded1 = (data1_all == -1).sum()

        n_unique = len(np.unique(np.concatenate([data0, data1])))
        is_discrete = n_unique <= 10

        if is_discrete:
            all_unique = sorted(set(data0) | set(data1))

            if len(data0) > 0:
                unique0, counts0 = np.unique(data0, return_counts=True)
                percent_dict0 = dict(zip(unique0, (counts0 / len(data0)) * 100))
                percent0 = [percent_dict0.get(val, 0) for val in all_unique]
            else:
                percent0 = [0] * len(all_unique)

            if len(data1) > 0:
                unique1, counts1 = np.unique(data1, return_counts=True)
                percent_dict1 = dict(zip(unique1, (counts1 / len(data1)) * 100))
                percent1 = [percent_dict1.get(val, 0) for val in all_unique]
            else:
                percent1 = [0] * len(all_unique)

            min_percent_threshold = 0.5
            filtered_indices = [
                i for i, val in enumerate(all_unique)
                if percent0[i] >= min_percent_threshold or percent1[i] >= min_percent_threshold
            ]

            if not filtered_indices:
                combined_percent = [max(p0, p1) for p0, p1 in zip(percent0, percent1)]
                filtered_indices = sorted(range(len(combined_percent)),
                                        key=lambda i: combined_percent[i],
                                        reverse=True)[:10]

            all_unique = [all_unique[i] for i in filtered_indices]
            percent0 = [percent0[i] for i in filtered_indices]
            percent1 = [percent1[i] for i in filtered_indices]

            x_labels = [str(int(val)) if val == int(val) else str(val) for val in all_unique]

            fig.add_trace(go.Bar(
                x=x_labels,
                y=percent0,
                name='Bez phishingu',
                marker_color=colors[0],
                opacity=0.85,
                showlegend=(i == 0),
                offsetgroup=0,
                hovertemplate='<b>Wartość:</b> %{x}<br><b>Procent:</b> %{y:.1f}%<extra></extra>'
            ), row=row, col=col)

            fig.add_trace(go.Bar(
                x=x_labels,
                y=percent1,
                name='Phishing',
                marker_color=colors[1],
                opacity=0.85,
                showlegend=(i == 0),
                offsetgroup=1,
                hovertemplate='<b>Wartość:</b> %{x}<br><b>Procent:</b> %{y:.1f}%<extra></extra>'
            ), row=row, col=col)

            max_percent = max(max(percent0, default=0), max(percent1, default=0))
            y_max = max_percent * 1.25 if max_percent > 0 else 100

            if y_max <= 1:
                dtick = 0.2
            elif y_max <= 5:
                dtick = 1
            elif y_max <= 10:
                dtick = 2
            elif y_max <= 50:
                dtick = 5
            elif y_max <= 100:
                dtick = 10
            else:
                dtick = 20

            fig.update_xaxes(
                title_text='Wartość cechy',
                row=row, col=col,
                title_font=dict(size=15, color='black', family='Arial Black'),
                tickfont=dict(size=13, color='black', family='Arial Black'),
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray',
                type='category',
                tickmode='linear',
                tickangle=0 if len(all_unique) <= 3 else -45,
                showline=True,
                linewidth=2,
                linecolor='black'
            )

            fig.update_yaxes(
                title_text='Procent [%]',
                row=row, col=col,
                title_font=dict(size=15, color='black', family='Arial Black'),
                tickfont=dict(size=12, color='black'),
                range=[0, y_max],
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray',
                dtick=dtick,
                tick0=0,
                showline=True,
                linewidth=2,
                linecolor='black',
                automargin=True
            )
        else:
            fig.add_trace(go.Histogram(
                x=data0,
                name='Bez phishingu',
                marker_color=colors[0],
                opacity=0.6,
                showlegend=(i == 0),
                histnorm='percent',
                nbinsx=25,
                hovertemplate='<b>Zakres:</b> %{x}<br><b>Procent:</b> %{y:.1f}%<extra></extra>'
            ), row=row, col=col)

            fig.add_trace(go.Histogram(
                x=data1,
                name='Phishing',
                marker_color=colors[1],
                opacity=0.6,
                showlegend=(i == 0),
                histnorm='percent',
                nbinsx=25,
                hovertemplate='<b>Zakres:</b> %{x}<br><b>Procent:</b> %{y:.1f}%<extra></extra>'
            ), row=row, col=col)

            all_data = np.concatenate([data0, data1])
            x_min, x_max = all_data.min(), all_data.max()
            x_range = x_max - x_min
            if x_range > 1000:
                nticks = 10
            elif x_range > 100:
                nticks = 12
            elif x_range > 10:
                nticks = 15
            else:
                nticks = 20

            fig.update_xaxes(
                title_text='Wartość cechy',
                row=row, col=col,
                title_font=dict(size=15, color='black', family='Arial Black'),
                tickfont=dict(size=13, color='black', family='Arial Black'),
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray',
                range=[x_min - 0.05 * x_range, x_max + 0.05 * x_range],
                nticks=nticks,
                showline=True,
                linewidth=2,
                linecolor='black'
            )

            fig.update_yaxes(
                title_text='Procent [%]',
                row=row, col=col,
                title_font=dict(size=15, color='black', family='Arial Black'),
                tickfont=dict(size=12, color='black'),
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray',
                tick0=0,
                showline=True,
                linewidth=2,
                linecolor='black',
                automargin=True,
                rangemode='tozero'
            )

    fig.update_layout(
        title_text='Rozkłady Top 8 Cech według Klas (najważniejsze wartości, pominięto -1)',
        title_font=dict(size=20),
        height=1200,
        width=2400,
        barmode='group',
        legend_title_text='Klasa',
        legend=dict(font=dict(size=15), x=1.01, y=0.5),
        font=dict(size=11),
        showlegend=True,
        bargap=0.15,
        bargroupgap=0.05,
        plot_bgcolor='white',
        uniformtext_minsize=8,
        uniformtext_mode='hide'
    )
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=12, color='black')

    fig.write_html(f'{html_dir}/distributions_top8_features.html')
    fig.write_image(f'{png_dir}/distributions_top8_features.png', scale=2, width=2400, height=1200)
    fig.show()

    print("\n" + "=" * 80)
    print("WIZUALIZACJA WARTOŚCI -1 (NIE WYSTĘPUJE W URL)")
    print("=" * 80)

    missing_data = []
    for feature in top_features:
        data0_all = df[df['phishing'] == 0][feature]
        data1_all = df[df['phishing'] == 1][feature]
        excluded0 = (data0_all == -1).sum()
        excluded1 = (data1_all == -1).sum()

        if excluded0 > 0 or excluded1 > 0:
            readable_name = get_readable_name(feature)
            missing_data.append({
                'Cecha': readable_name,
                'Bez phishingu': (excluded0 / len(data0_all)) * 100,
                'Phishing': (excluded1 / len(data1_all)) * 100,
                'Bez phishingu (count)': excluded0,
                'Phishing (count)': excluded1
            })
            print(f"  {readable_name}:")
            print(f"    • Bez phishingu: {excluded0} ({excluded0/len(data0_all)*100:.1f}%)")
            print(f"    • Phishing: {excluded1} ({excluded1/len(data1_all)*100:.1f}%)")

    if missing_data:
        missing_df = pd.DataFrame(missing_data)

        fig_missing = go.Figure()

        fig_missing.add_trace(go.Bar(
            name='Bez phishingu',
            x=missing_df['Cecha'],
            y=missing_df['Bez phishingu'],
            marker_color='#636EFA',
            hovertemplate='<b>%{x}</b><br>Nie występuje w URL: %{y:.1f}%<br>Liczba: %{customdata}<extra></extra>',
            customdata=missing_df['Bez phishingu (count)']
        ))

        fig_missing.add_trace(go.Bar(
            name='Phishing',
            x=missing_df['Cecha'],
            y=missing_df['Phishing'],
            marker_color='#EF553B',
            hovertemplate='<b>%{x}</b><br>Nie występuje w URL: %{y:.1f}%<br>Liczba: %{customdata}<extra></extra>',
            customdata=missing_df['Phishing (count)']
        ))

        fig_missing.update_layout(
            title='Procent Przypadków gdzie Cecha Nie Występuje w URL (wartości -1)',
            title_font=dict(size=18, color='black', family='Arial Black'),
            xaxis_title='Cecha',
            yaxis_title='Procent przypadków z wartością -1 [%]',
            xaxis_title_font=dict(size=14, color='black', family='Arial Black'),
            yaxis_title_font=dict(size=14, color='black', family='Arial Black'),
            barmode='group',
            height=500,
            width=1400,
            showlegend=True,
            legend_title_text='Klasa',
            legend=dict(font=dict(size=12)),
            xaxis=dict(
                tickangle=-45,
                tickfont=dict(size=11, color='black'),
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray',
                showline=True,
                linewidth=2,
                linecolor='black'
            ),
            yaxis=dict(
                tickfont=dict(size=12, color='black'),
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray',
                showline=True,
                linewidth=2,
                linecolor='black',
                range=[0, max(missing_df['Bez phishingu'].max(), missing_df['Phishing'].max()) * 1.15]
            ),
            plot_bgcolor='white',
            bargap=0.2,
            bargroupgap=0.1
        )

        fig_missing.write_html(f'{html_dir}/missing_values_minus1.html')
        fig_missing.write_image(f'{png_dir}/missing_values_minus1.png', scale=2, width=1400, height=500)
        fig_missing.show()
        print(f"\n✓ Wizualizacja nieobecności cech wygenerowana")
        print(f"✓ Wizualizacja zapisana do: {png_dir}/missing_values_minus1.png")
    else:
        print("  ✅ Brak wartości -1 w top 8 cechach")

    normality_df.to_csv(f'{csv_dir}/distribution_analysis.csv', index=False)
    print(f"\n✓ Analiza rozkładu zapisana do: {csv_dir}/distribution_analysis.csv")
    print(f"✓ Wizualizacja rozkładów zapisana do: {png_dir}/distributions_top8_features.png")

    return normality_df


distribution_analysis = analyze_distributions(df_clean)


ANALIZA ROZKŁADÓW

Sprawdzanie wartości -1 (nie występuje w URL):
   • Znaleziono 0 cech z wartościami -1
   • Wartość -1 oznacza: cecha nie występuje w URL

Wyniki Testu Normalności (Shapiro-Wilk, α=0.05):
   • Cechy z rozkładem normalnym: 38 (36.5%)
   • Cechy z rozkładem nienormalnym: 66



WIZUALIZACJA WARTOŚCI -1 (NIE WYSTĘPUJE W URL)
  ✅ Brak wartości -1 w top 8 cechach

✓ Analiza rozkładu zapisana do: ../static/statistical_analysis_url_corpus/csv/distribution_analysis.csv
✓ Wizualizacja rozkładów zapisana do: ../static/statistical_analysis_url_corpus/png/distributions_top8_features.png


In [ ]:
def generate_missing_minus_one_artifacts(df, top_features, html_dir, png_dir):
    missing_rows = []
    for feature in top_features:
        data0_all = df[df['phishing'] == 0][feature]
        data1_all = df[df['phishing'] == 1][feature]
        excluded0 = int((data0_all == -1).sum())
        excluded1 = int((data1_all == -1).sum())

        missing_rows.append({
            'Cecha': get_readable_name(feature),
            'Bez phishingu': (excluded0 / len(data0_all)) * 100 if len(data0_all) else 0.0,
            'Phishing': (excluded1 / len(data1_all)) * 100 if len(data1_all) else 0.0,
            'Bez phishingu (count)': excluded0,
            'Phishing (count)': excluded1,
        })

    missing_df = pd.DataFrame(missing_rows)

    fig_missing = go.Figure()
    fig_missing.add_trace(go.Bar(
        name='Bez phishingu',
        x=missing_df['Cecha'],
        y=missing_df['Bez phishingu'],
        marker_color='#636EFA',
        hovertemplate='<b>%{x}</b><br>Nie występuje w URL: %{y:.1f}%<br>Liczba: %{customdata}<extra></extra>',
        customdata=missing_df['Bez phishingu (count)']
    ))
    fig_missing.add_trace(go.Bar(
        name='Phishing',
        x=missing_df['Cecha'],
        y=missing_df['Phishing'],
        marker_color='#EF553B',
        hovertemplate='<b>%{x}</b><br>Nie występuje w URL: %{y:.1f}%<br>Liczba: %{customdata}<extra></extra>',
        customdata=missing_df['Phishing (count)']
    ))

    y_max = float(max(missing_df['Bez phishingu'].max(), missing_df['Phishing'].max()))
    if y_max <= 0:
        y_max = 1.0

    fig_missing.update_layout(
        title='Procent Przypadków gdzie Cecha Nie Występuje w URL (wartości -1)',
        title_font=dict(size=18, color='black', family='Arial Black'),
        xaxis_title='Cecha',
        yaxis_title='Procent przypadków z wartością -1 [%]',
        xaxis_title_font=dict(size=14, color='black', family='Arial Black'),
        yaxis_title_font=dict(size=14, color='black', family='Arial Black'),
        barmode='group',
        height=500,
        width=1400,
        showlegend=True,
        legend_title_text='Klasa',
        legend=dict(font=dict(size=12)),
        xaxis=dict(
            tickangle=-45,
            tickfont=dict(size=11, color='black'),
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showline=True,
            linewidth=2,
            linecolor='black'
        ),
        yaxis=dict(
            tickfont=dict(size=12, color='black'),
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showline=True,
            linewidth=2,
            linecolor='black',
            range=[0, y_max * 1.15]
        ),
        plot_bgcolor='white',
        bargap=0.2,
        bargroupgap=0.1
    )

    html_path = f'{html_dir}/missing_values_minus1.html'
    png_path = f'{png_dir}/missing_values_minus1.png'
    fig_missing.write_html(html_path)
    fig_missing.write_image(png_path, scale=2, width=1400, height=500)
    print(f'✓ Wymuszony zapis wykresu -1: {png_path}')
    print(f'✓ Wymuszony zapis HTML -1: {html_path}')


top_features_for_missing = feature_importance['Cecha'].head(8).tolist()
generate_missing_minus_one_artifacts(df_clean, top_features_for_missing, html_dir, png_dir)


✓ Wymuszony zapis wykresu -1: ../static/statistical_analysis_url_corpus/png/missing_values_minus1.png
✓ Wymuszony zapis HTML -1: ../static/statistical_analysis_url_corpus/html/missing_values_minus1.html


## 7. Wizualizacja - Mapa Ciepła Korelacji


In [9]:
def create_correlation_heatmap(df, top_n=25):
    print("\nTworzenie interaktywnej mapy ciepła korelacji...")

    target_col = df.columns[-1]
    top_features = target_correlations.head(top_n).index.tolist()

    corr_matrix = df[top_features + [target_col]].corr()

    readable_labels = [get_readable_name(col) if col != target_col else col for col in corr_matrix.columns]

    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=readable_labels,
        y=readable_labels,
        colorscale='RdBu_r',
        zmin=-1, zmax=1,
        text=corr_matrix.round(2).values,
        texttemplate="%{text}",
        hoverongaps=False
    ))

    fig.update_layout(
        title=f'Interaktywna Mapa Ciepła Korelacji - Top {top_n} Cech',
        width=900, height=900,
        xaxis_tickangle=-45
    )
    fig.show()

    fig.write_image(f'{png_dir}/correlation_heatmap.png', scale=2)
    print(f"✓ Mapa ciepła zapisana do: {png_dir}/correlation_heatmap.png")

    return corr_matrix

correlation_heatmap = create_correlation_heatmap(df_clean, top_n=25)


Tworzenie interaktywnej mapy ciepła korelacji...


✓ Mapa ciepła zapisana do: ../static/statistical_analysis_url_corpus/png/correlation_heatmap.png


## 8. Zaawansowana Analiza Selekcji Cech


In [10]:
def advanced_feature_selection_analysis(df, feature_importance_df, correlation_matrix, target_corr_df):
    print("=" * 80)
    print("ZAAWANSOWANA ANALIZA SELEKCJI CECH")
    print("=" * 80)

    feature_cols = [col for col in df.columns if col != 'phishing']
    features_to_eliminate = {}

    zero_variance = df[feature_cols].var()[df[feature_cols].var() == 0].index.tolist()
    near_zero_variance = df[feature_cols].var()[(df[feature_cols].var() > 0) & (df[feature_cols].var() < 0.01)].index.tolist()

    features_to_eliminate['zero_variance'] = zero_variance
    features_to_eliminate['near_zero_variance'] = near_zero_variance

    print(f"\n1. WARIANCJA:")
    print(f"   • Cechy o zerowej wariancji: {len(zero_variance)}")
    print(f"   • Cechy o niemal zerowej wariancji (<0.01): {len(near_zero_variance)}")

    corr_threshold = 0.95
    high_corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > corr_threshold:
                feature_i = correlation_matrix.columns[i]
                feature_j = correlation_matrix.columns[j]

                importance_i = feature_importance_df[feature_importance_df['Cecha'] == feature_i]['Ważność'].values[0] if feature_i in feature_importance_df['Cecha'].values else 0
                importance_j = feature_importance_df[feature_importance_df['Cecha'] == feature_j]['Ważność'].values[0] if feature_j in feature_importance_df['Cecha'].values else 0

                eliminate_feature = feature_i if importance_i < importance_j else feature_j
                high_corr_pairs.append((feature_i, feature_j, correlation_matrix.iloc[i, j], eliminate_feature))

    highly_correlated = list(set([pair[3] for pair in high_corr_pairs]))
    features_to_eliminate['high_correlation'] = highly_correlated

    print(f"\n2. WYSOKA KORELACJA (>{corr_threshold}):")
    print(f"   • Znaleziono par silnie skorelowanych: {len(high_corr_pairs)}")
    print(f"   • Cechy do eliminacji (niższa ważność): {len(highly_correlated)}")

    low_importance = feature_importance_df[feature_importance_df['Ważność'] < 0.001]['Cecha'].tolist()
    features_to_eliminate['low_importance'] = low_importance

    print(f"\n3. WAŻNOŚĆ CECH (Random Forest):")
    print(f"   • Cechy o bardzo niskiej ważności (<0.001): {len(low_importance)}")

    weak_target_corr = target_corr_df[target_corr_df['Abs_Korelacja'] < 0.05].index.tolist()
    features_to_eliminate['weak_target_correlation'] = weak_target_corr

    print(f"\n4. KORELACJA Z ZMIENNĄ DOCELOWĄ:")
    print(f"   • Cechy o słabej korelacji z 'phishing' (<0.05): {len(weak_target_corr)}")

    skewness = df[feature_cols].skew()
    high_skewness = skewness[abs(skewness) > 5].index.tolist()

    kurtosis = df[feature_cols].kurtosis()
    high_kurtosis = kurtosis[abs(kurtosis) > 10].index.tolist()

    print(f"\n5. ROZKŁADY:")
    print(f"   • Cechy o wysokiej skośności (|skewness| > 5): {len(high_skewness)}")
    print(f"   • Cechy o wysokiej kurtozie (|kurtosis| > 10): {len(high_kurtosis)}")

    all_eliminate = list(set(
        zero_variance + near_zero_variance + highly_correlated +
        low_importance + weak_target_corr
    ))

    elimination_counts = {feat: sum([1 for cat_feats in features_to_eliminate.values() if feat in cat_feats])
                         for feat in all_eliminate}

    high_confidence_eliminate = [feat for feat, count in elimination_counts.items() if count >= 2]

    print(f"\n6. PODSUMOWANIE ELIMINACJI:")
    print("=" * 80)
    print(f"   • Łączna liczba unikalnych cech do potencjalnej eliminacji: {len(all_eliminate)}")
    print(f"   • Cechy do eliminacji (wysoka pewność - ≥2 powody): {len(high_confidence_eliminate)}")
    print(f"   • Cechy do przeglądu (średnia pewność - 1 powód): {len(all_eliminate) - len(high_confidence_eliminate)}")

    high_importance = feature_importance_df.nlargest(30, 'Ważność')['Cecha'].tolist()
    strong_correlation = target_corr_df[target_corr_df['Abs_Korelacja'] >= 0.3].index.tolist()

    strong_features = set(high_importance) & set(strong_correlation)
    strong_features = [f for f in strong_features if f not in all_eliminate]

    print(f"   • Cechy z wysoką ważnością I silną korelacją: {len(strong_features)}")
    if strong_features:
        readable_strong = [get_readable_name(f) for f in list(strong_features)[:20]]
        print(f"     Top cechy: {', '.join(readable_strong)}")

    print("\n7. KOŃCOWE REKOMENDACJE SELEKCJI CECH:")
    print("=" * 80)

    features_to_keep_final = [f for f in feature_cols if f not in high_confidence_eliminate]

    print(f"   • Oryginalna liczba cech: {len(feature_cols)}")
    print(f"   • Rekomendowane do ELIMINACJI (wysoka pewność): {len(high_confidence_eliminate)}")
    print(f"   • Rekomendowane do ZACHOWANIA: {len(features_to_keep_final)}")
    print(f"   • Redukcja cech: {len(high_confidence_eliminate)/len(feature_cols)*100:.1f}%")

    elimination_df = pd.DataFrame([
        {'Cecha': feat, 'Pewność_Eliminacji': 'Wysoka' if feat in high_confidence_eliminate else 'Średnia',
         'Liczba_Powodów': elimination_counts.get(feat, 0),
         'Powody': ', '.join([cat for cat, feats in features_to_eliminate.items() if feat in feats])}
        for feat in all_eliminate
    ]).sort_values('Liczba_Powodów', ascending=False)

    elimination_df.to_csv(f'{csv_dir}/features_to_eliminate.csv', index=False)
    print(f"\n✓ Rekomendacje eliminacji zapisane do: {csv_dir}/features_to_eliminate.csv")

    keep_df = pd.DataFrame({
        'Cecha': features_to_keep_final,
        'Ważność_RF': [feature_importance_df[feature_importance_df['Cecha'] == f]['Ważność'].values[0]
                          if f in feature_importance_df['Cecha'].values else 0
                          for f in features_to_keep_final],
        'Korelacja_z_Celem': [target_corr_df.loc[f, 'Korelacja']
                               if f in target_corr_df.index else 0
                               for f in features_to_keep_final]
    }).sort_values('Ważność_RF', ascending=False)

    keep_df.to_csv(f'{csv_dir}/features_to_keep.csv', index=False)
    print(f"✓ Cechy do zachowania zapisane do: {csv_dir}/features_to_keep.csv")

    return elimination_df, keep_df, high_confidence_eliminate, features_to_keep_final

elimination_recommendations, features_to_keep_df, eliminate_list, keep_list = advanced_feature_selection_analysis(
    df_clean, feature_importance, correlation_matrix, target_correlations
)

ZAAWANSOWANA ANALIZA SELEKCJI CECH

1. WARIANCJA:
   • Cechy o zerowej wariancji: 29
   • Cechy o niemal zerowej wariancji (<0.01): 14

2. WYSOKA KORELACJA (>0.95):
   • Znaleziono par silnie skorelowanych: 2
   • Cechy do eliminacji (niższa ważność): 2

3. WAŻNOŚĆ CECH (Random Forest):
   • Cechy o bardzo niskiej ważności (<0.001): 70

4. KORELACJA Z ZMIENNĄ DOCELOWĄ:
   • Cechy o słabej korelacji z 'phishing' (<0.05): 24

5. ROZKŁADY:
   • Cechy o wysokiej skośności (|skewness| > 5): 62
   • Cechy o wysokiej kurtozie (|kurtosis| > 10): 68

6. PODSUMOWANIE ELIMINACJI:
   • Łączna liczba unikalnych cech do potencjalnej eliminacji: 73
   • Cechy do eliminacji (wysoka pewność - ≥2 powody): 53
   • Cechy do przeglądu (średnia pewność - 1 powód): 20
   • Cechy z wysoką ważnością I silną korelacją: 13
     Top cechy: SSL: Czy ma ważny certyfikat, URL: Liczba "-" w całym URL, Plik: Liczba ".", Ścieżka: Liczba "/", URL: Liczba "=" w całym URL, URL: Liczba "/" w całym URL, HTTP: Liczba przekie

## 9. Tworzenie Zoptymalizowanego Zbioru Danych i Podsumowanie


In [11]:
def create_optimized_dataset_and_summarize(df, keep_list, eliminate_list):
    print("=" * 80)
    print("TWORZENIE ZOPTYMALIZOWANEGO ZBIORU DANYCH I PODSUMOWANIE")
    print("=" * 80)

    target_col = df.columns[-1]

    df_optimized = df[keep_list + [target_col]].copy()

    print(f"\n1. ZBIÓR DANYCH - SELEKCJA CECH (Feature Selection):")
    print(f"   • Oryginalna liczba cech: {df.shape[1] - 1}")
    print(f"   • Cechy po eliminacji: {len(keep_list)}")
    print(f"   • Usunięte cechy: {len(eliminate_list)}")
    print(f"   • Redukcja: {len(eliminate_list) / (df.shape[1] - 1) * 100:.1f}%")

    optimized_path = '../data/url_corpus/dataset_optimized.csv'
    df_optimized.to_csv(optimized_path, index=False)
    print(f"   ✓ Zapisano do: {optimized_path}")

    print(f"\n" + "=" * 80)
    print("PORÓWNANIE ZBIORÓW DANYCH")
    print("=" * 80)

    comparison_data = {
        'Zbiór danych': [
            'Full (pełny)',
            'Optimized (eliminacja cech)'
        ],
        'Liczba cech': [
            df.shape[1] - 1,
            len(keep_list)
        ],
        'Typ cech': [
            'Wszystkie atrybuty (URL + DNS/WHOIS)',
            'Wszystkie atrybuty (URL + DNS/WHOIS)'
        ],
        'Interpretowalność': [
            'Wysoka',
            'Wysoka'
        ],
        'Plik': [
            'dataset_full.csv',
            'dataset_optimized.csv'
        ]
    }

    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))

    summary = f"""
# Podsumowanie Kompleksowej Analizy Statystycznej

## 1. Przegląd Danych
- **Liczba próbek:** {len(df):,}
- **Liczba cech (oryginalnie):** {df.shape[1] - 1}
- **Typ cech:** URL + DNS/WHOIS (wszystkie reprodukowalne)
- **Rozkład klas:**
    - Phishing: {df[target_col].value_counts()[1]:,} ({df[target_col].value_counts(normalize=True)[1]:.2%})
    - Bezpieczne: {df[target_col].value_counts()[0]:,} ({df[target_col].value_counts(normalize=True)[0]:.2%})

## 2. Utworzony Zbiór Danych

### Optimized (Selekcja Cech)
- **Plik:** `dataset_optimized.csv`
- **Liczba cech:** {len(keep_list)} (z {df.shape[1] - 1})
- **Redukcja:** {len(eliminate_list) / (df.shape[1] - 1):.1%}
- **Typ:** Wszystkie atrybuty (URL + DNS/WHOIS)
- **Zalety:**
  - Zachowuje oryginalne cechy - łatwo zrozumieć znaczenie
  - Dobra do analizy biznesowej i wyjaśniania modelu
  - Zawiera cechy z URLFeatureAnalyzer (szybkie) + AdvancedURLAnalyzer (wolne)
  - Najwyższa dokładność modeli
- **Zastosowanie:** Modele wymagające maksymalnej dokładności

## 3. Kluczowe Cechy (Feature Selection)
**Top 5 najważniejszych zachowanych cech:**
"""
    for i, feature in enumerate(feature_importance['Cecha'].head(5), 1):
        importance = feature_importance[feature_importance['Cecha'] == feature]['Ważność'].values[0]
        summary += f"{i}. `{feature}` (ważność: {importance:.4f})\n"

    summary += f"""
## 4. Rekomendacje

### Wybór Zbioru Danych:
- **Dataset Full** → Maksymalna informacja (111 cech)
- **Dataset Optimized** → Najlepszy stosunek jakości do wydajności ({len(keep_list)} cech)

### Zalecane Modele:
- Random Forest, XGBoost, Logistic Regression (dobre do interpretacji)
- SVM, Neural Networks (dla lepszej dokładności)

## 5. Informacja o Reprodukowalności Cech
Wszystkie cechy są reprodukowalne przy użyciu:
- **URLFeatureAnalyzer** - szybka ekstrakcja z URL (89 cech URL-only)
- **AdvancedURLAnalyzer** - wolna ekstrakcja z DNS/WHOIS/HTTP (11 cech):
  - time_response, domain_spf, asn_ip
  - time_domain_activation, time_domain_expiration
  - qty_ip_resolved, qty_nameservers, qty_mx_servers
  - ttl_hostname, url_google_index, domain_google_index

## 6. Następne Kroki
1. Przetestować oba zbiory danych z różnymi modelami
2. Porównać dokładność i czas trenowania
3. Wybrać najlepsze podejście dla aplikacji

## 7. Zapisane Artefakty
- **Dane oryginalne:** `../data/url_corpus/dataset_full.csv` ({df.shape[1] - 1} cech)
- **Dane zoptymalizowane:** `../data/url_corpus/dataset_optimized.csv` ({len(keep_list)} cech)
- **Wizualizacje PNG:** `../static/statistical_analysis_url_corpus/png/`
- **Wizualizacje HTML:** `../static/statistical_analysis_url_corpus/html/`
- **Raporty CSV:** `../static/statistical_analysis_url_corpus/csv/`
- **Dokumentacja:** `../static/statistical_analysis_url_corpus/docs/`
"""

    summary_path = f'{docs_dir}/analysis_summary.md'
    with open(summary_path, 'w') as f:
        f.write(summary)

    print(f"\n2. PODSUMOWANIE ANALIZY ZAPISANE:")
    print(f"   ✓ Lokalizacja: {summary_path}")

    print(f"\n" + "=" * 80)
    print("REKOMENDACJE UŻYCIA:")
    print("=" * 80)
    print("• dataset_full.csv → Maksymalna dokładność, 111 cech (URL + DNS/WHOIS)")
    print("• dataset_optimized.csv → Wysoka interpretowalność, optymalna wydajność")
    print("\n✅ Wszystkie cechy są reprodukowalne (URLFeatureAnalyzer + AdvancedURLAnalyzer)")

    return df_optimized

df_optimized = create_optimized_dataset_and_summarize(df_clean, keep_list, eliminate_list)

TWORZENIE ZOPTYMALIZOWANEGO ZBIORU DANYCH I PODSUMOWANIE

1. ZBIÓR DANYCH - SELEKCJA CECH (Feature Selection):
   • Oryginalna liczba cech: 104
   • Cechy po eliminacji: 51
   • Usunięte cechy: 53
   • Redukcja: 51.0%
   ✓ Zapisano do: ../data/url_corpus/dataset_optimized.csv

PORÓWNANIE ZBIORÓW DANYCH
               Zbiór danych  Liczba cech                             Typ cech Interpretowalność                  Plik
               Full (pełny)          104 Wszystkie atrybuty (URL + DNS/WHOIS)            Wysoka      dataset_full.csv
Optimized (eliminacja cech)           51 Wszystkie atrybuty (URL + DNS/WHOIS)            Wysoka dataset_optimized.csv

2. PODSUMOWANIE ANALIZY ZAPISANE:
   ✓ Lokalizacja: ../static/statistical_analysis_url_corpus/docs/analysis_summary.md

REKOMENDACJE UŻYCIA:
• dataset_full.csv → Maksymalna dokładność, 111 cech (URL + DNS/WHOIS)
• dataset_optimized.csv → Wysoka interpretowalność, optymalna wydajność

✅ Wszystkie cechy są reprodukowalne (URLFeatureAnalyzer 

## 10. Podsumowanie Zapisanych Plików


In [12]:
import os
from pathlib import Path

print("=" * 80)
print("PODSUMOWANIE WSZYSTKICH ZAPISANYCH PLIKÓW")
print("=" * 80)

saved_files = {
    'Zbiory danych': [
        '../data/url_corpus/dataset_full.csv',
        '../data/url_corpus/dataset_optimized.csv'
    ],
    'Raporty CSV': [
        f'{csv_dir}/descriptive_statistics.csv',
        f'{csv_dir}/target_correlations.csv',
        f'{csv_dir}/feature_importance.csv',
        f'{csv_dir}/distribution_analysis.csv',
        f'{csv_dir}/features_to_eliminate.csv',
        f'{csv_dir}/features_to_keep.csv'
    ],
    'Wizualizacje (PNG)': [
        f'{png_dir}/target_correlations.png',
        f'{png_dir}/feature_importance.png',
        f'{png_dir}/distributions_top8_features.png',
        f'{png_dir}/missing_values_minus1.png',
        f'{png_dir}/correlation_heatmap.png'
    ],
    'Wizualizacje (HTML - interaktywne)': [
        f'{html_dir}/target_correlations.html',
        f'{html_dir}/feature_importance.html',
        f'{html_dir}/distributions_top8_features.html',
        f'{html_dir}/missing_values_minus1.html'
    ],
    'Dokumentacja': [
        f'{docs_dir}/analysis_summary.md'
    ]
}

for category, files in saved_files.items():
    print(f"\n{category}:")
    for file_path in files:
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path)
            if file_size > 1024*1024:
                size_str = f"{file_size/(1024*1024):.2f} MB"
            elif file_size > 1024:
                size_str = f"{file_size/1024:.2f} KB"
            else:
                size_str = f"{file_size} B"
            print(f"  ✅ {file_path} ({size_str})")
        else:
            print(f"  ❌ {file_path} (nie znaleziono)")

print("\n" + "=" * 80)
print("STRUKTURA KATALOGÓW")
print("=" * 80)
print(f"📁 Dane:")
print(f"   └─ ../data/")
print(f"      ├─ dataset_full.csv")
print(f"      └─ dataset_optimized.csv")
print(f"\n📊 Analiza statystyczna: {output_dir}/")
print(f"   ├─ 📄 csv/          (raporty CSV)")
print(f"   ├─ 🖼️  png/          (obrazy statyczne)")
print(f"   ├─ 🌐 html/         (wykresy interaktywne)")
print(f"   └─ 📝 docs/         (dokumentacja)")
print(f"\n✅ Analiza zakończona pomyślnie!")


PODSUMOWANIE WSZYSTKICH ZAPISANYCH PLIKÓW

Zbiory danych:
  ✅ ../data/url_corpus/dataset_full.csv (22.42 MB)
  ✅ ../data/url_corpus/dataset_optimized.csv (9.31 MB)

Raporty CSV:
  ✅ ../static/statistical_analysis_url_corpus/csv/descriptive_statistics.csv (8.42 KB)
  ✅ ../static/statistical_analysis_url_corpus/csv/target_correlations.csv (4.76 KB)
  ✅ ../static/statistical_analysis_url_corpus/csv/feature_importance.csv (3.20 KB)
  ✅ ../static/statistical_analysis_url_corpus/csv/distribution_analysis.csv (5.39 KB)
  ✅ ../static/statistical_analysis_url_corpus/csv/features_to_eliminate.csv (4.43 KB)
  ✅ ../static/statistical_analysis_url_corpus/csv/features_to_keep.csv (2.88 KB)

Wizualizacje (PNG):
  ✅ ../static/statistical_analysis_url_corpus/png/target_correlations.png (177.71 KB)
  ✅ ../static/statistical_analysis_url_corpus/png/feature_importance.png (287.16 KB)
  ✅ ../static/statistical_analysis_url_corpus/png/distributions_top8_features.png (441.79 KB)
  ✅ ../static/statistical_ana